In [18]:
# from google.colab import drive
# drive.mount('/content/drive')

In [19]:
import math
import os
from typing import Any

import flax.nnx as nnx
import grain.python as grain
import jax
import jax.numpy as jnp
import numpy as np
import optax
import orbax.checkpoint as ocp

from datasets import Dataset, load_dataset
from transformers import AutoTokenizer, PreTrainedTokenizer
from grain.checkpoint import CheckpointSave as GrainCheckpointSave, CheckpointRestore as GrainCheckpointRestore

In [20]:
"""
Gated DeltaNet-2 (GDN-2) — JAX / Flax NNX implementation
Paper: https://arxiv.org/abs/2605.22791
"""

import jax
import jax.numpy as jnp
from flax import nnx

# ---------------------------------------------------------------------------
# Core recurrence (mathematically correct in the original; kept intact)
# Parameter names are now aligned with paper notation:
#   b_t = erase gate  (paper: b_t ∈ [0,1]^{d_k})
#   w_t = write gate  (paper: w_t ∈ [0,1]^{d_v})
#   alpha_t = per-channel decay  (paper: α_t ∈ (0,1]^{d_k})
# ---------------------------------------------------------------------------


def chunked_forward(
    query: jax.Array,  # (B, L, d_k)
    key: jax.Array,  # (B, L, d_k)
    value: jax.Array,  # (B, L, d_v)
    b: jax.Array,  # (B, L, d_k) — erase gate
    w: jax.Array,  # (B, L, d_v) — write gate
    alpha: jax.Array,  # (B, L, d_k) — per-channel decay  ← was "delta"
    chunk_size: int,
) -> jax.Array:
    """
    Implements the Gated Delta Rule-2 recurrence (paper Eq. 10):

        S_t = (I − k_t (b_t ⊙ k_t)ᵀ) Diag(α_t) S_{t-1} + k_t (w_t ⊙ v_t)ᵀ
        o_t = Sₜᵀ q_t

    Uses a two-level loop:
      outer — jax.lax.scan across chunks (sequential across chunk boundaries)
      inner — jax.lax.associative_scan inside each chunk (parallel prefix)
    """
    batch_size, seq_len, dk = query.shape
    dv = value.shape[-1]
    num_chunks = seq_len // chunk_size
    assert seq_len % chunk_size == 0, (
        f"seq_len ({seq_len}) must be divisible by chunk_size ({chunk_size})"
    )

    # Reshape → (num_chunks, batch, chunk_size, dim) so scan iterates axis-0
    def to_scan(x):
        return x.reshape(batch_size, num_chunks, chunk_size, -1).swapaxes(0, 1)

    q_s = to_scan(query)
    k_s = to_scan(key)
    v_s = to_scan(value)
    b_s = to_scan(b)
    w_s = to_scan(w)
    a_s = to_scan(alpha)  # renamed from d_s

    I = jnp.eye(dk, dtype=query.dtype)

    # ── Associative scan combiner ──────────────────────────────────────────
    # Encodes the linear recurrence S_t = A_t S_{t-1} + B_t
    # (A1,B1) = prefix ending at t1; (A2,B2) = update from t1+1 to t2
    def combine(left, right):
        A1, B1 = left
        A2, B2 = right
        return A2 @ A1, A2 @ B1 + B2

    # ── Outer scan: one step = one chunk ──────────────────────────────────
    def chunk_step(S_prev, xs):
        q_c, k_c, v_c, b_c, w_c, a_c = xs

        # ── Build per-token A and B matrices (paper Eq. 10) ──────────────
        # erase factor: e_t = b_t ⊙ k_t
        # A_t = (I − k_t eₜᵀ) Diag(α_t)
        #     = (I − k_t (b_t⊙k_t)ᵀ) Diag(α_t)
        # [i,j] = (δ_{ij} − k_i·(b⊙k)_j) · α_j   ← column-wise scaling by α
        e_c = b_c * k_c  # (B, C, dk)
        outer_ke = jnp.einsum("bci,bcj->bcij", k_c, e_c)  # k eᵀ
        # Expand α to (B,C,1,dk) so it broadcasts column-wise
        A_c = (I - outer_ke) * jnp.expand_dims(a_c, axis=-2)  # (B,C,dk,dk)

        # write term: B_t = k_t (w_t ⊙ v_t)ᵀ = k_t zₜᵀ
        z_c = w_c * v_c  # (B, C, dv)
        B_c = jnp.einsum("bci,bcj->bcij", k_c, z_c)  # (B,C,dk,dv)

        # ── Associative scan over the chunk (parallel prefix) ─────────────
        A_ct = A_c.swapaxes(0, 1)  # (C, B, dk, dk)
        B_ct = B_c.swapaxes(0, 1)  # (C, B, dk, dv)
        A_cum_t, B_cum_t = jax.lax.associative_scan(combine, (A_ct, B_ct))
        A_cum = A_cum_t.swapaxes(0, 1)  # (B, C, dk, dk)
        B_cum = B_cum_t.swapaxes(0, 1)  # (B, C, dk, dv)

        # ── Compute all hidden states in this chunk ───────────────────────
        # S_r = A_cum_r S_prev + B_cum_r
        S_all = jnp.einsum("bcij,bjk->bcik", A_cum, S_prev) + B_cum
        # (B, C, dk, dv)

        # ── Outputs: o_r = Sᵀ_r q_r ─────────────────────────────────────
        o_c = jnp.einsum("bci,bcij->bcj", q_c, S_all)  # (B, C, dv)

        S_next = S_all[:, -1, :]  # last token's state
        return S_next, o_c

    S_init = jnp.zeros((batch_size, dk, dv), dtype=query.dtype)
    _, o_chunks = jax.lax.scan(
        chunk_step, S_init, (q_s, k_s, v_s, b_s, w_s, a_s)
    )  # o_chunks: (num_chunks, batch, chunk_size, dv)

    o = o_chunks.swapaxes(0, 1)  # (batch, num_chunks, chunk_size, dv)
    return o.reshape(batch_size, num_chunks * chunk_size, dv)


# ---------------------------------------------------------------------------
# Short causal convolution helper  [B3]
# Paper Fig.1: q, k, v each pass through a short causal conv + SiLU
# ---------------------------------------------------------------------------


class ShortCausalConv(nnx.Module):
    """
    Depthwise causal convolution with kernel size 4 (as used in GDN-2 / GDN).
    Equivalent to FLA's ShortConvolution in casual mode.
    """

    def __init__(self, dim: int, kernel_size: int = 4, rngs: nnx.Rngs = None):
        # Depthwise: groups = dim
        self.kernel_size = kernel_size
        self.dim = dim
        # Weight: (dim, 1, kernel_size)
        self.weight = nnx.Param(
            jax.random.normal(rngs.params(), (dim, kernel_size)) * 0.02
        )

    def __call__(self, x: jax.Array) -> jax.Array:
        # x: (B, L, dim)
        B, L, D = x.shape
        k = self.kernel_size
        # Causal padding: pad k-1 zeros at the start of the time axis
        x_pad = jnp.pad(x, ((0, 0), (k - 1, 0), (0, 0)))  # (B, L+k-1, D)
        # Depthwise conv via sliding window
        # weight: (D, k),  x_windows: (B, L, D, k)
        x_windows = jnp.stack(
            [x_pad[:, i : i + L, :] for i in range(k)], axis=-1
        )  # (B, L, D, k)
        out = jnp.einsum("bldk,dk->bld", x_windows, self.weight)
        return out


# ---------------------------------------------------------------------------
# Gated DeltaNet-2 token mixer  (GDN-2 block, Fig. 1)
# ---------------------------------------------------------------------------


class GatedDeltaNet2Layer(nnx.Module):
    """
    Full GDN-2 token mixer matching paper Figure 1 and Section 3.5:

      q, k paths: Linear → ShortConv → SiLU → L2-norm
      v   path:   Linear → ShortConv → SiLU
      α  (decay): dedicated projection → exp(-exp(a) * softplus(proj + bias))
      b  (erase): Linear → sigmoid
      w  (write): Linear → sigmoid
      output:     recurrent output → RMSNorm → SiLU-gate → Linear
    """

    def __init__(
        self,
        dim: int,
        num_heads: int,
        chunk_size: int,
        conv_kernel: int = 4,
        rngs: nnx.Rngs = None,
    ):
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.chunk_size = chunk_size

        # ── Linear projections ──────────────────────────────────────────
        self.q_proj = nnx.Linear(dim, dim, use_bias=False, rngs=rngs)
        self.k_proj = nnx.Linear(dim, dim, use_bias=False, rngs=rngs)
        self.v_proj = nnx.Linear(dim, dim, use_bias=False, rngs=rngs)

        # ── Short causal convolutions  [B3] ────────────────────────────
        self.q_conv = ShortCausalConv(dim, conv_kernel, rngs=rngs)
        self.k_conv = ShortCausalConv(dim, conv_kernel, rngs=rngs)
        self.v_conv = ShortCausalConv(dim, conv_kernel, rngs=rngs)

        # ── Gate projections ────────────────────────────────────────────
        # b_t (erase): d_model → d_k per head, sigmoid → [0,1]^{d_k}
        self.b_proj = nnx.Linear(dim, dim, use_bias=False, rngs=rngs)
        # w_t (write): d_model → d_v per head, sigmoid → [0,1]^{d_v}
        # [B5] renamed from gamma_proj
        self.w_proj = nnx.Linear(dim, dim, use_bias=False, rngs=rngs)

        # ── Decay projection  [B2][B5] ──────────────────────────────────
        # Paper Eq. 12:  g_t = −exp(a) ⊙ softplus(W_f x_t + δ_bias)
        #                α_t = exp(g_t)  ∈ (0,1]^{d_k}
        # We store the learnable per-head-channel log-scale `a` separately.
        self.decay_proj = nnx.Linear(dim, dim, use_bias=True, rngs=rngs)
        # Learnable log-scale per key channel: shape (num_heads, head_dim)
        self.decay_log_scale = nnx.Param(jnp.zeros((num_heads, self.head_dim)))

        # ── Output gate + norm  [B4] ────────────────────────────────────
        # Paper Sec 3.5: "output is RMS-normalised, multiplied by a SiLU
        # output gate, and projected back to the model dimension"
        self.out_norm = nnx.RMSNorm(dim, rngs=rngs)
        self.out_gate_proj = nnx.Linear(dim, dim, use_bias=False, rngs=rngs)
        self.o_proj = nnx.Linear(dim, dim, use_bias=False, rngs=rngs)

    # ── helpers ──────────────────────────────────────────────────────────

    def _to_heads(self, x: jax.Array, batch: int, seq: int) -> jax.Array:
        """(B, L, dim) → (B*H, L, d_k)  for the flat recurrence call."""
        # (B, L, H, D_h) → (B, H, L, D_h) → (B*H, L, D_h)
        return (
            x.reshape(batch, seq, self.num_heads, self.head_dim)
            .swapaxes(1, 2)
            .reshape(batch * self.num_heads, seq, self.head_dim)
        )

    def _from_heads(self, x: jax.Array, batch: int, seq: int) -> jax.Array:
        """(B*H, L, d_h) → (B, L, dim)"""
        return (
            x.reshape(batch, self.num_heads, seq, self.head_dim)
            .swapaxes(1, 2)
            .reshape(batch, seq, self.num_heads * self.head_dim)
        )

    @staticmethod
    def _l2_norm(x: jax.Array, eps: float = 1e-6) -> jax.Array:
        return x / jnp.maximum(jnp.linalg.norm(x, axis=-1, keepdims=True), eps)

    # ── forward ──────────────────────────────────────────────────────────

    def __call__(self, x: jax.Array) -> jax.Array:
        B, L, _ = x.shape

        # ── q, k, v:  proj → conv → SiLU → L2 (q, k only)  [B1][B3] ──
        q = jax.nn.silu(self.q_conv(self.q_proj(x)))
        k = jax.nn.silu(self.k_conv(self.k_proj(x)))
        v = jax.nn.silu(self.v_conv(self.v_proj(x)))

        # L2-normalise both q AND k per head  [B1]
        q_h = q.reshape(B, L, self.num_heads, self.head_dim)
        k_h = k.reshape(B, L, self.num_heads, self.head_dim)
        q_h = self._l2_norm(q_h)
        k_h = self._l2_norm(k_h)

        # ── Erase gate b_t ∈ [0,1]^{d_k}  ─────────────────────────────
        b_h = jax.nn.sigmoid(self.b_proj(x)).reshape(
            B, L, self.num_heads, self.head_dim
        )

        # ── Write gate w_t ∈ [0,1]^{d_v}  [B5] ────────────────────────
        w_h = jax.nn.sigmoid(self.w_proj(x)).reshape(
            B, L, self.num_heads, self.head_dim
        )
        v_h = v.reshape(B, L, self.num_heads, self.head_dim)

        # ── Decay α_t ∈ (0,1]^{d_k}  [B2][B5] ──────────────────────────
        # Paper Eq.12:  g_t = −exp(a) ⊙ softplus(W_f x_t + bias)
        #               α_t = exp(g_t)
        raw = self.decay_proj(x)  # (B, L, dim)
        raw_h = raw.reshape(B, L, self.num_heads, self.head_dim)
        # decay_log_scale: (H, D_h), broadcast over B and L
        log_scale = jnp.exp(self.decay_log_scale)  # exp(a) ∈ (0,∞)
        # Compute g_t in higher precision for numerical stability (App D.1)
        g = -log_scale * jax.nn.softplus(raw_h.astype(jnp.float32))
        alpha_h = jnp.exp(g).astype(x.dtype)  # α_t ∈ (0,1]

        # ── Flatten (B,L,H,D_h) → (B*H, L, D_h) for the recurrence ────
        def flat(t):
            # t: (B, L, H, D_h)
            return t.swapaxes(1, 2).reshape(B * self.num_heads, L, self.head_dim)

        q_flat = flat(q_h)
        k_flat = flat(k_h)
        v_flat = flat(v_h)
        b_flat = flat(b_h)
        w_flat = flat(w_h)
        a_flat = flat(alpha_h)

        # ── Core recurrence ─────────────────────────────────────────────
        o_flat = chunked_forward(
            q_flat, k_flat, v_flat, b_flat, w_flat, a_flat, self.chunk_size
        )  # (B*H, L, D_h)

        # ── Recombine heads → (B, L, dim) ───────────────────────────────
        o = self._from_heads(o_flat, B, L)

        # ── Output: RMSNorm → SiLU-gate → projection  [B4] ─────────────
        # Paper Sec 3.5: output = (RMSNorm(o) * SiLU(gate)) W_o
        o_normed = self.out_norm(o)
        gate = jax.nn.silu(self.out_gate_proj(x))
        return self.o_proj(o_normed * gate)


# ---------------------------------------------------------------------------
# Transformer-style block (pre-norm, SwiGLU MLP)
# ---------------------------------------------------------------------------


class GatedDeltaNet2Block(nnx.Module):
    def __init__(
        self,
        dim: int,
        num_heads: int,
        mlp_dim: int,
        chunk_size: int,
        rngs: nnx.Rngs,
    ):
        self.norm1 = nnx.RMSNorm(dim, rngs=rngs)
        self.norm2 = nnx.RMSNorm(dim, rngs=rngs)
        self.mixer = GatedDeltaNet2Layer(dim, num_heads, chunk_size, rngs=rngs)
        # SwiGLU MLP
        self.mlp_gate = nnx.Linear(dim, mlp_dim, use_bias=False, rngs=rngs)
        self.mlp_up = nnx.Linear(dim, mlp_dim, use_bias=False, rngs=rngs)
        self.mlp_down = nnx.Linear(mlp_dim, dim, use_bias=False, rngs=rngs)

    def __call__(self, x: jax.Array) -> jax.Array:
        x = x + self.mixer(self.norm1(x))
        h = self.norm2(x)
        x = x + self.mlp_down(jax.nn.silu(self.mlp_gate(h)) * self.mlp_up(h))
        return x


# ---------------------------------------------------------------------------
# Full language model
# ---------------------------------------------------------------------------


class GatedDeltaNet2(nnx.Module):
    def __init__(
        self,
        vocab_size: int,
        num_layers: int,
        dim: int,
        num_heads: int,
        chunk_size: int,
        rngs: nnx.Rngs,
    ):
        self.embed = nnx.Embed(vocab_size, dim, rngs=rngs)
        self.blocks = nnx.Sequential(
            *[
                GatedDeltaNet2Block(
                    dim, num_heads, mlp_dim=dim * 4, chunk_size=chunk_size, rngs=rngs
                )
                for _ in range(num_layers)
            ]
        )
        self.norm_f = nnx.RMSNorm(dim, rngs=rngs)
        self.lm_head = nnx.Linear(dim, vocab_size, use_bias=False, rngs=rngs)

    def __call__(self, input_ids: jax.Array) -> jax.Array:
        x = self.embed(input_ids)
        x = self.blocks(x)
        x = self.norm_f(x)
        return self.lm_head(x)

In [21]:
# --- Wrap Hugging Face Dataset in a Grain Data Source ---
class HuggingFaceDataSource(grain.RandomAccessDataSource):
    """
    A Grain wrapper for Hugging Face datasets.
    Because HF relies on Apache Arrow under the hood, random lookups are incredibly fast.
    """
    def __init__(self, hf_ds: Dataset) -> None:
        self.hf_ds = hf_ds

    def __len__(self) -> int:
        return len(self.hf_ds)

    def __getitem__(self, index: int) -> dict[str, Any]:
        # HF natively returns a dictionary for the row (e.g., {'text': 'Some string'})
        return self.hf_ds[index]

In [22]:
class TokenizerAndShift(grain.MapTransform):
    def __init__(self, tokenizer: PreTrainedTokenizer, max_length: int = 128) -> None:
        self.tokenizer = tokenizer
        self.max_length = max_length

    def map(self, element: dict[str, Any]) -> dict[str, Any]:
        encoded: dict[str, list[int]] = self.tokenizer(
            element["text"],
            truncation=True,
            max_length=self.max_length + 1,  # +1 for the shift
            padding="max_length",
            return_tensors=None,
        )

        tokens: list[int] = encoded["input_ids"]

        new_element = {
            "inputs": tokens[:-1],
            "targets": tokens[1:],
            "attention_mask": encoded["attention_mask"][:-1],
        }

        return new_element

In [23]:
class ConvertToJaxArrays(grain.MapTransform):
    def map(self, element: dict[str, Any]) -> dict[str, Any]:
        for key in ["inputs", "targets", "attention_mask"]:
            element[key] = jnp.array(np.array(element[key]))
        return element

In [24]:
class FilterEmptyLines(grain.FilterTransform):
    def filter(self, element: dict[str, Any]) -> bool:
        return len(element["text"].strip()) > 0

In [25]:
def build_dataloader(
    dataset: Dataset,
    tokenizer: PreTrainedTokenizer,
    batch_size: int = 8,
    max_length: int = 128,
) -> grain.DataLoader:
    source = HuggingFaceDataSource(dataset)

    sampler = grain.IndexSampler(
        num_records=len(source),
        num_epochs=1,
        shard_options=grain.ShardOptions(
            shard_index=0, shard_count=1, drop_remainder=True
        ),
        shuffle=True,
        seed=42,
    )

    loader = grain.DataLoader(
        data_source=source,
        sampler=sampler,
        operations=[
            FilterEmptyLines(),
            TokenizerAndShift(tokenizer, max_length=max_length),
            ConvertToJaxArrays(),
            grain.Batch(batch_size=batch_size, drop_remainder=True),
        ],
        worker_count=0,
    )

    return loader

In [26]:
def loss_fn(model: nnx.Module, batch: dict[str, jax.Array]) -> jax.Array:
    logits = model(batch["inputs"])
    loss = optax.softmax_cross_entropy_with_integer_labels(
        logits=logits, labels=batch["targets"]
    ).mean()

    return loss

In [27]:
@nnx.jit
def train_step(
    model: nnx.Module, optimizer: nnx.Optimizer, batch: dict[str, jax.Array]
) -> jax.Array:
    grad_fn = nnx.value_and_grad(loss_fn)
    loss, grads = grad_fn(model, batch)

    optimizer.update(model, grads)
    return loss

In [28]:
@nnx.jit
def eval_step(model: nnx.Module, batch: dict[str, jax.Array]) -> jax.Array:
    logits = model(batch["inputs"])
    loss = optax.softmax_cross_entropy_with_integer_labels(
        logits=logits, labels=batch["targets"]
    ).mean()

    return loss

In [29]:
def save_checkpoint(
    mngr: ocp.CheckpointManager,
    step: int,
    model: nnx.Module,
    optimizer: nnx.Optimizer,
    rngs: nnx.Rngs,
    data_iterator
) -> None:
    """Bundles model weights, optimizer momentum, and Grain iterator state to disk."""
    print(f"Saving checkpoint at step {step}...")

    # 1. Get the NNX state (Model + Optimizer)
    # _, nnx_state = nnx.split((model, optimizer))

    # 2. Get the Grain iterator state
    # grain_state = data_iterator.get_state()

    # 3. Create a unified PyTree dictionary
    unified_state = {
        "model": ocp.args.StandardSave(model),
        "optimizer": ocp.args.StandardSave(optimizer),
        "rngs": ocp.args.StandardSave(rngs),
    }

    # 4. Save the unified state
    mngr.save(
        step,
        args=ocp.args.Composite(**unified_state)
    )

    # Block until save is complete to ensure safety
    mngr.wait_until_finished()
    print("Save complete!")

In [30]:
def restore_checkpoint(
    mngr: ocp.CheckpointManager,
    model: nnx.Module,
    optimizer: nnx.Optimizer,
    rngs: nnx.Rngs,
    data_iterator
) -> int:
    """Restores the unified state and injects it back into the objects."""

    latest_step = mngr.latest_step()
    if latest_step is None:
        print("No existing checkpoints found. Starting from scratch (Step 0).")
        return 0

    print(f"Found checkpoint at step {latest_step}. Restoring...")

    # 1. Create the abstract template for NNX
    # _, abstract_nnx_state = nnx.split((model, optimizer))

    # 2. Create the abstract template for Grain
    # (We can just use its current empty state as the template)
    # abstract_grain_state = data_iterator.get_state()

    # 3. Assemble the abstract unified state
    abstract_unified_state = {
        "model": ocp.args.StandardRestore(model),
        "optimizer": ocp.args.StandardRestore(optimizer),
        "rngs": ocp.args.StandardRestore(rngs),
    }

    # 4. Restore from disk
    restored_state = mngr.restore(
        latest_step,
        args=ocp.args.Composite(**abstract_unified_state)
    )

    # 5. Inject the restored states back into their respective objects
    # nnx.update((model, optimizer), restored_state["nnx"])
    # data_iterator.set_state(restored_state["grain"])

    print("Restore complete! Model, Optimizer, and DataLoader are synchronized.")
    return latest_step

In [31]:
@nnx.jit
def predict_next_token(model: nnx.Module, input_ids: jax.Array) -> jax.Array:
    """Runs a single forward pass to predict the next word."""

    # 1. Get the raw scores (logits) for every token in the vocabulary
    logits = model(input_ids)

    # 2. Isolate the predictions for the very last token in our sequence
    # Shape goes from (batch, seq_len, vocab_size) -> (vocab_size,)
    next_token_logits = logits[0, -1, :]

    # 3. Greedy Decoding: Simply pick the token with the highest probability score
    next_token = jnp.argmax(next_token_logits)

    return next_token

In [32]:
def interactive_chat(model: nnx.Module, tokenizer: PreTrainedTokenizer, max_new_tokens: int = 100):
    """Starts an infinite loop for user interaction."""

    print("\n" + "="*50)
    print("🤖 Massive LLM is online and ready!")
    print("Type 'quit' or 'exit' to shut down the server.")
    print("="*50 + "\n")

    # The Infinite Loop
    while True:
        # 1. Get User Input
        user_text = input("You: ")

        # 2. Check for exit commands
        if user_text.strip().lower() in ['quit', 'exit']:
            print("Shutting down the model. Goodbye!")
            break

        # Skip empty inputs
        if not user_text.strip():
            continue

        # 3. Tokenize the input into a standard NumPy array
        # We add batch dimension manually so shape is (1, seq_len)
        encoded = tokenizer(user_text, return_tensors="np")
        input_ids = encoded["input_ids"]

        print("Model: ", end="", flush=True)

        # 4. The Autoregressive Generation Loop
        for _ in range(max_new_tokens):

            # A. Predict the next token
            next_token_array = predict_next_token(model, input_ids)

            # Convert the JAX array back to a standard Python integer
            next_token_id = next_token_array.item()

            # B. Check for the End-Of-Sequence (EOS) token
            # If the model decides it is done talking, break the generation loop
            if next_token_id == tokenizer.eos_token_id:
                break

            # C. Decode the single integer back into a readable word
            word = tokenizer.decode([next_token_id])

            # Print the word immediately (flush=True forces the terminal to update)
            print(word, end="", flush=True)

            # D. Append the new token to our sequence so the model can read it
            # on the next loop iteration.
            input_ids = np.append(input_ids, [[next_token_id]], axis=1)

        # Print a newline when the model finishes its complete thought
        print("\n")

In [33]:
def train_and_evaluate(num_epochs: int = 1000, eval_every_n_steps: int = 5, save_every_n_steps: int = 5):
    train_dataset: Dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
    val_dataset: Dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="validation")

    gpt2tokenizer: PreTrainedTokenizer = AutoTokenizer.from_pretrained("gpt2")
    gpt2tokenizer.pad_token = gpt2tokenizer.eos_token

    train_loader: grain.DataLoader = build_dataloader(
        train_dataset, gpt2tokenizer, batch_size=8, max_length=128
    )
    val_loader: grain.DataLoader = build_dataloader(
        val_dataset, gpt2tokenizer, batch_size=8, max_length=128
    )

    # Initialize Model and Optimizer

    rngs: nnx.Rngs = nnx.Rngs(0)
    model: nnx.Module = GatedDeltaNet2(
        vocab_size=gpt2tokenizer.vocab_size,
        num_layers=6,
        dim=512,
        num_heads=8,
        chunk_size=16,
        rngs=rngs
    )

    """ GPTModel(
        vocab_size=gpt2tokenizer.vocab_size,
        embed_dim=512,
        num_q_heads=8,
        num_kv_heads=4,
        head_dim=64,
        seq_length=128,
        dropout_rate=0.1,
        n_layers=6,
        emb_dim_multiply=4,
        rngs=rngs,
    ) """
    optimizer = nnx.Optimizer(model, optax.adamw(learning_rate=3e-4), wrt=nnx.Param)

    # 1. Define the directory path
    CHECKPOINT_DIR = ''

    """try:
        from google.colab import drive
        drive.mount('/content/drive')
        CHECKPOINT_DIR = os.path.abspath("/content/drive/My Drive/nugie_llm_checkpoints")
    except ImportError:
        CHECKPOINT_DIR = os.path.abspath("./nugie_llm_checkpoints")"""

    # 2. Configure the manager's behavior
    # options = ocp.CheckpointManagerOptions(max_to_keep=1, create=True)

    # 3. Initialize the 'mngr' variable
    # mngr = ocp.CheckpointManager(CHECKPOINT_DIR, options=options)

    data_iterator = iter(train_loader)

    step = 0 # restore_checkpoint(mngr, model, optimizer, rngs, data_iterator)

    print("Starting training...")
    if step > 0:
        print(f"Resuming training from step {step}...")

    for epoch in range(num_epochs):
        for batch in data_iterator:
            train_loss = train_step(model, optimizer, batch)

            if step % eval_every_n_steps == 0 and step > 0:
                total_val_loss = 0.0
                val_steps = 0

                for val_batch in val_loader:
                    val_loss = eval_step(model, val_batch)
                    total_val_loss += val_loss
                    val_steps += 1

                avg_val_loss = total_val_loss / val_steps
                perplexity = math.exp(avg_val_loss)

                print(
                    f"Val Loss: {avg_val_loss:.4f} | Perplexity: {perplexity:.2f} | Epoch: {epoch + 1}/{num_epochs}"
                )

            print(
                f"Step {step:04d} | Train Loss: {train_loss:.4f} | Epoch: {epoch + 1}/{num_epochs}"
            )
            step += 1

            # if step % save_every_n_steps == 0:
            #     save_checkpoint(mngr, step, model, optimizer, rngs, data_iterator)

        step = 0  # Reset step count after each epoch

    interactive_chat(model, gpt2tokenizer, max_new_tokens=150)

In [34]:
import sys

from absl import flags

# flags.FLAGS(sys.argv)  # Parse flags to keep Grain multiprocessing happy

train_and_evaluate()

Starting training...
Step 0000 | Train Loss: 11.2245 | Epoch: 1/1000
Step 0001 | Train Loss: 11.5597 | Epoch: 1/1000


KeyboardInterrupt: 